# Evaluate Nova Models on SWE-bench

[SWE-bench](https://www.swebench.com/) evaluates a model's ability to resolve real GitHub issues by writing code patches. Each sample runs in a Docker container with the repository code.

## Prerequisites

1. **Docker installed and running** — [Install Docker](https://docs.docker.com/get-started/get-docker/)
2. **AWS credentials configured**
3. **Python 3.10+**
4. **~5 GB disk space** for Docker images

**Time:** ~30-60 min | **Cost:** Bedrock per-token charges (SWE-bench uses many tokens per sample)

## Setup

In [ ]:
%pip install "inspect-ai>=0.3.220" "inspect-evals[swe_bench]" --quiet

In [ ]:
import subprocess, shutil, boto3

if shutil.which("docker"):
    result = subprocess.run(["docker", "info"], capture_output=True, text=True)
    print("\u2713 Docker running" if result.returncode == 0 else "\u2717 Docker not running")
else:
    print("\u2717 Docker not found")

try:
    print(f"\u2713 AWS credentials ({boto3.client('sts').get_caller_identity()['Account']})")
except:
    print("\u2717 AWS credentials not configured")

## Run SWE-bench

Uses `swe_bench_verified_mini` (50 samples) with `--limit 5`. Each sample pulls a Docker image (~1 GB), presents a GitHub issue to the model, and verifies the fix passes tests.

**Note:** SWE-bench requires tool-calling support (Nova Pro, Claude). Nova Micro does not work.

In [ ]:
# Run SWE-bench (5 samples). Change --model if using a SageMaker endpoint.
!inspect eval inspect_evals/swe_bench_verified_mini \
    --model bedrock/us.amazon.nova-pro-v1:0 \
    -M region_name=us-east-1 \
    --limit 5 \
    --max-connections 1 \
    --max-retries 3 \
    --display plain

In [ ]:
!inspect view

## Customization

```bash
# All 50 samples
inspect eval inspect_evals/swe_bench_verified_mini --model bedrock/us.amazon.nova-pro-v1:0

# Full SWE-bench Verified (500 samples, ~130 GB Docker images)
inspect eval inspect_evals/swe_bench --model bedrock/us.amazon.nova-pro-v1:0

# Increase parallelism
inspect eval inspect_evals/swe_bench_verified_mini --model bedrock/us.amazon.nova-pro-v1:0 --max-connections 3
```